# WalkThru — video/photos → walkable 3D house (MESH pipeline, v2)

Our own Polycam-style reconstruction on Colab's free T4 GPU.
**v2 lessons baked in:** video input, resumable cells, Drive checkpoints after every heavy stage, no COLMAP trimmer (it crashes — we crop with trimesh), Y-flip + bowl-crop done here so you download a small ready GLB.

**Before starting:**
1. Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save
2. Put your input in Drive folder `MyDrive/WalkThru/`: either a **video** (.mp4/.mov walkthrough) or a **zip of photos**
3. Run cells in order; each ends with ✅. If one fails, send its full output back for debugging.

⚠️ CELL 2 restarts the runtime — the 'session crashed' popup there is NORMAL. Continue from CELL 3.

For **Gaussian splats**, run the separate `WalkThru_Colab_Splats.ipynb` AFTER this one (it reuses this run's Drive checkpoint, in a FRESH runtime).

In [ ]:
# CELL 1 — confirm GPU
!nvidia-smi | head -12
print("✅ CELL 1 done — a Tesla T4 (or similar) should be listed above")

In [ ]:
# CELL 2 — install conda. RUNTIME RESTARTS AFTER THIS ('session crashed' popup = NORMAL).
# After the restart continue from CELL 3. Do NOT rerun this cell.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# CELL 3 — install COLMAP with CUDA (~3-5 min)
import os, shutil, subprocess
if shutil.which("mamba") is None:
    raise RuntimeError("mamba missing -> CELL 2 has not completed in THIS runtime. Run CELL 2, wait for the restart, then rerun this cell.")
pin = "/usr/local/conda-meta/pinned"
if os.path.exists(pin):
    print("removing bogus pin:", open(pin).read())
    os.remove(pin)
r = subprocess.run(["mamba","install","-y","-q","-c","conda-forge","colmap","libfaiss","openimageio"], capture_output=True, text=True)
print(r.stdout[-800:]); print(r.stderr[-500:])
assert r.returncode == 0, "mamba install FAILED — send this output"
v = subprocess.run(["colmap","-h"], capture_output=True, text=True)
out = (v.stdout or "") + (v.stderr or "")
print("\n".join(out.splitlines()[:6]))
if "cannot open shared object" in out:
    print(subprocess.run(["bash","-lc","ldd /usr/local/bin/colmap | grep 'not found'"], capture_output=True, text=True).stdout)
assert "COLMAP" in out and "cannot open shared object" not in out, "colmap does not run — send this output"
print("✅ CELL 3 done — colmap runs (header should say 'with CUDA')")

In [ ]:
# CELL 4 — INPUT: set EXACTLY ONE of VIDEO / PHOTO_ZIP, then run
RUN_NAME  = "myroom"            # <-- name this property/scan (used for Drive folders)
VIDEO     = ""                   # e.g. "/content/drive/MyDrive/WalkThru/room.mp4"
PHOTO_ZIP = "/content/drive/MyDrive/WalkThru/walkthru_photos_test5.zip"
TARGET_FRAMES = 90               # for video: how many frames to extract (60-120 is good)

from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, zipfile, subprocess, shutil

WORK = "/content/recon"; IMAGES = f"{WORK}/images"
DRIVE_RUN = f"/content/drive/MyDrive/WalkThru/runs/{RUN_NAME}"
os.makedirs(IMAGES, exist_ok=True); os.makedirs(DRIVE_RUN, exist_ok=True)
intr = None
IS_VIDEO = bool(VIDEO)
assert bool(VIDEO) != bool(PHOTO_ZIP), "Set exactly ONE of VIDEO or PHOTO_ZIP (leave the other as '')"

if IS_VIDEO:
    dur = float(subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",VIDEO], capture_output=True, text=True).stdout.strip())
    fps = max(0.2, TARGET_FRAMES / dur)
    print(f"video: {dur:.1f}s -> extracting at {fps:.2f} fps")
    r = subprocess.run(["ffmpeg","-y","-i",VIDEO,"-vf",f"fps={fps:.4f},scale='min(1600,iw)':-2","-q:v","2",f"{IMAGES}/img_%04d.jpg"], capture_output=True, text=True)
    assert r.returncode == 0, "ffmpeg failed — send this output:\n" + r.stderr[-800:]
else:
    with zipfile.ZipFile(PHOTO_ZIP) as z: z.extractall(IMAGES)
    subs = [d for d in glob.glob(f"{IMAGES}/*") if os.path.isdir(d)]
    if subs and not glob.glob(f"{IMAGES}/*.jpg"):
        for f in glob.glob(subs[0] + "/*"): os.rename(f, f"{IMAGES}/" + os.path.basename(f))
    # intrinsics.json must NOT sit in the images dir (COLMAP tries to read it as an image)
    ip = f"{IMAGES}/intrinsics.json"
    if os.path.exists(ip):
        shutil.move(ip, f"{WORK}/intrinsics.json")
        intr = json.load(open(f"{WORK}/intrinsics.json", encoding="utf-8-sig"))

imgs = sorted(glob.glob(f"{IMAGES}/*.jpg")) + sorted(glob.glob(f"{IMAGES}/*.png"))
print(f"frames ready: {len(imgs)} | video mode: {IS_VIDEO} | intrinsics: {intr}")
assert 25 <= len(imgs) <= 300, f"{len(imgs)} frames — aim for 40-150 (adjust TARGET_FRAMES)"
print("✅ CELL 4 done")

In [ ]:
# CELL 5 — features + matching on GPU (~1-5 min)
import subprocess, time, shutil, os
assert shutil.which("colmap"), "colmap missing -> runtime was recycled. Rerun CELLS 2,3,4 in order."
DB = f"{WORK}/db.db"

def run(name, args, allow_fail=False):
    t = time.time(); print(f"=== {name} ===", flush=True)
    r = subprocess.run(args, capture_output=True, text=True)
    print(((r.stdout or "")[-900:] + "\n" + (r.stderr or "")[-900:]).strip())
    if r.returncode != 0 and not allow_fail: raise RuntimeError(f"{name} FAILED exit {r.returncode}")
    print(f"--- {name}: {time.time()-t:.0f}s (exit {r.returncode})")
    return r.returncode

def flag(cmd, new, old):
    h = subprocess.run(["colmap",cmd,"--help"], capture_output=True, text=True)
    return new if new.lstrip('-') in ((h.stdout or "")+(h.stderr or "")) else old
EX_GPU = flag("feature_extractor","--FeatureExtraction.use_gpu","--SiftExtraction.use_gpu")
MA_GPU = flag("exhaustive_matcher","--FeatureMatching.use_gpu","--SiftMatching.use_gpu")

cam = []
if intr: cam = ["--ImageReader.camera_model", intr["model"], "--ImageReader.camera_params", ",".join(str(p) for p in intr["params"])]
elif IS_VIDEO: cam = ["--ImageReader.camera_model", "OPENCV"]

run("extract (GPU)", ["colmap","feature_extractor","--database_path",DB,"--image_path",IMAGES,"--ImageReader.single_camera","1",*cam,EX_GPU,"1"])
if IS_VIDEO:
    run("match sequential (GPU)", ["colmap","sequential_matcher","--database_path",DB,"--SequentialMatching.overlap","20",MA_GPU,"1"])
else:
    run("match exhaustive (GPU)", ["colmap","exhaustive_matcher","--database_path",DB,MA_GPU,"1"])
print("✅ CELL 5 done")

In [ ]:
# CELL 6 — solve camera poses (SfM, CPU ~5-20 min) + checkpoint to Drive
import os, subprocess
SPARSE = f"{WORK}/sparse"; os.makedirs(SPARSE, exist_ok=True)
margs = ["colmap","mapper","--database_path",DB,"--image_path",IMAGES,"--output_path",SPARSE]
if not os.path.isdir(f"{SPARSE}/0"):
    rc = run("SfM", margs, allow_fail=True)
    if rc != 0 or not os.path.isdir(f"{SPARSE}/0"):
        run("SfM retry (relaxed)", margs + ["--Mapper.init_min_num_inliers","50","--Mapper.init_min_tri_angle","4"])
else:
    print("sparse/0 already exists — skipping (resumable)")
run("model report", ["colmap","model_analyzer","--path",f"{SPARSE}/0"])
# checkpoint: images + sparse to Drive (also the input for the SPLATS notebook)
subprocess.run(["bash","-lc",f"cp -r {SPARSE} '{DRIVE_RUN}/' && cp -r {IMAGES} '{DRIVE_RUN}/' && cp {DB} '{DRIVE_RUN}/'"], check=True)
print("checkpointed images+sparse+db to", DRIVE_RUN)
print("✅ CELL 6 done — SEND ME: Registered images / Points / Mean reprojection error")

In [ ]:
# CELL 7 — undistort (seconds, resumable)
DENSE = f"{WORK}/dense"
import os
if not os.path.isdir(f"{DENSE}/images"):
    run("undistort", ["colmap","image_undistorter","--image_path",IMAGES,"--input_path",f"{SPARSE}/0","--output_path",DENSE,"--output_type","COLMAP"])
else:
    print("dense workspace exists — skipping")
print("✅ CELL 7 done")

In [ ]:
# CELL 8 — GPU dense depth. QUALITY: 'draft' ~15 min, 'full' ~60-70 min (measured on T4, 140 imgs)
QUALITY = "full"   # "draft" while testing a new capture, "full" for the real model
opts = {"draft": ["--PatchMatchStereo.max_image_size","1000","--PatchMatchStereo.geom_consistency","false"],
        "full":  ["--PatchMatchStereo.max_image_size","1600","--PatchMatchStereo.geom_consistency","true"]}[QUALITY]
run("patch match stereo (GPU)", ["colmap","patch_match_stereo","--workspace_path",DENSE,"--workspace_format","COLMAP","--PatchMatchStereo.gpu_index","0","--PatchMatchStereo.cache_size","12",*opts])
print("✅ CELL 8 done")

In [ ]:
# CELL 9 — fuse depth into a colored point cloud (~5-8 min) + checkpoint to Drive
import subprocess, os
ftype = "geometric" if QUALITY == "full" else "photometric"
run("stereo fusion", ["colmap","stereo_fusion","--workspace_path",DENSE,"--workspace_format","COLMAP","--input_type",ftype,"--output_path",f"{DENSE}/fused.ply","--StereoFusion.cache_size","12"])
print(subprocess.run(["bash","-lc",f"ls -lh {DENSE}/fused.ply"], capture_output=True, text=True).stdout)
subprocess.run(["bash","-lc",f"cp {DENSE}/fused.ply '{DRIVE_RUN}/'"], check=True)
print("checkpointed fused.ply to Drive")
print("✅ CELL 9 done")

In [ ]:
# CELL 10 — Poisson mesh. NOTE: never use --PoissonMeshing.trim (the trimmer CRASHES in this build;
# we crop the bowl ourselves in CELL 11). ~10-15 min.
import os
if not os.path.exists(f"{DENSE}/mesh.ply"):
    run("poisson meshing", ["colmap","poisson_mesher","--input_path",f"{DENSE}/fused.ply","--output_path",f"{DENSE}/mesh.ply"])
else:
    print("mesh.ply exists — skipping")
print(subprocess.run(["bash","-lc",f"ls -lh {DENSE}/mesh.ply"], capture_output=True, text=True).stdout)
print("✅ CELL 10 done")

In [ ]:
# CELL 11 — clean + flip + export GLB (trimesh; do NOT use open3d here — it cannot import on this kernel)
!pip install -q trimesh scipy
import numpy as np, trimesh, subprocess, gc
m = trimesh.load(f"{DENSE}/mesh.ply", process=False)
print(f"raw: {len(m.vertices):,} verts {len(m.faces):,} faces")
# crop the Poisson 'bowl' hull: keep faces inside the dense-region percentile box
lo = np.percentile(m.vertices, 2, axis=0); hi = np.percentile(m.vertices, 98, axis=0)
pad = 0.15 * (hi - lo); lo, hi = lo - pad, hi + pad
cent = m.vertices[m.faces].mean(axis=1)
m.update_faces(np.all((cent >= lo) & (cent <= hi), axis=1)); m.remove_unreferenced_vertices()
labels = trimesh.graph.connected_component_labels(m.face_adjacency, node_count=len(m.faces))
counts = np.bincount(labels)
if len(counts) > 1 and counts.max() > 0.5 * len(m.faces):
    m.update_faces(labels == counts.argmax()); m.remove_unreferenced_vertices()
print(f"cleaned: {len(m.vertices):,} verts {len(m.faces):,} faces")
# COLMAP's world is Y-down -> flip 180° about X so the house is upright in the viewer
m.apply_transform(trimesh.transformations.rotation_matrix(np.pi, [1, 0, 0]))
GLB = f"/content/{RUN_NAME}_mesh.glb"
m.export(GLB); del m; gc.collect()
print(subprocess.run(["bash","-lc",f"ls -lh {GLB} && cp {GLB} '{DRIVE_RUN}/'"], capture_output=True, text=True).stdout)
print("✅ CELL 11 done — GLB saved to Drive runs folder")

In [ ]:
# CELL 12 — download the GLB (it is ALSO in Drive: WalkThru/runs/<RUN_NAME>/)
from google.colab import files
files.download(GLB)
print("✅ CELL 12 done")

## After downloading — on the laptop

The GLB is full resolution (hundreds of MB). Compress it for the web (proven: 478 MB → 0.4 MB):
```
cd D:\\startupidea
node node_modules\\@gltf-transform\\cli\\bin\\cli.js optimize <downloaded>.glb public\\scans\\myroom.glb --compress draco --simplify true --simplify-error 0.0001
```
Then walk it: `npm run dev` → `http://localhost:5173/?model=/scans/myroom.glb`

**Next level — Gaussian splats:** open `WalkThru_Colab_Splats.ipynb` in a **fresh runtime** (Runtime ▸ Disconnect and delete runtime first). It reuses this run's Drive checkpoint (`WalkThru/runs/<RUN_NAME>/`) — same photos, same camera solve — and trains a photoreal splat. The mesh from THIS notebook stays as the collision layer; the splat becomes the visual layer. That pairing is the product.